# 9 WorkFlow Analista Jr

### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

#### Parametros

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191

PARAM$experimento <- 9103
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [ ]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [ ]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

In [ ]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [ ]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [ ]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [ ]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [ ]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [ ]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [ ]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [ ]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [ ]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

In [ ]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [ ]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [ ]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [ ]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [ ]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [ ]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [ ]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [ ]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

In [ ]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


In [ ]:
colnames(dataset)

In [ ]:
# se intenta corregir el data drifting utilizando algunos indices financieros

#### 9.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [ ]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [ ]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [ ]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [ ]:
ncol(dataset)
colnames(dataset)

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  ni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 9.3.2 Modelado

#### 9.3.2.1 Training Strategy

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 201901, 202105 ]  donde se consideran el 100% de los CONTINUA

In [ ]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)


PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [ ]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [ ]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

In [ ]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [ ]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

####  9.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en la Bayesian Optimization (guia del profe)
  * learning_rate en escala log, rango [0.001, 1.5]
  * num_leaves en escala log, rango [2, 80000]
  * feature_fraction en escala lineal, rango [0.05, 1.0]
  * min_sum_hessian_in_leaf en escala log, rango [1e-8, 1000]
  * num_iterations se determina por early stopping en cada evaluacion

In [ ]:
# --- HIPERPARAMETROS FIJOS del LightGBM (guia del profe Denicolay) ---
# NO se optimizan porque su valor optimo esta establecido
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE,
  seed= PARAM$semilla_primigenia,
  # === fijos en default ===
  boosting= "gbdt",
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,
  max_depth= -1,
  lambda_l1= 0,
  lambda_l2= 0,
  bagging_fraction= 1.0,
  bagging_freq= 0,
  min_gain_to_split= 0,
  # === fijos en valor distinto al default ===
  min_data_in_leaf= 0,
  max_bin= 31,
  # === el tope de iteraciones se controla con early stopping ===
  num_iterations= 2048,
  early_stopping_rounds= 100
)

In [ ]:
# --- Instalo ParBayesianOptimization si no esta ---
if (!require("ParBayesianOptimization")) {
  install.packages("ParBayesianOptimization",
                   repos = "http://cran.us.r-project.org")
}
require("ParBayesianOptimization")

# Archivo de checkpoint incremental: se apenda una fila por cada evaluacion
PARAM$bo$checkpoint_file <- "bo_history_incremental.txt"

# --- Funcion objetivo para la Bayesian Optimization ---
# Recibe los hiperparametros a optimizar (algunos en escala LOG) y devuelve
# la AUC en validation. bayesOpt() la va a maximizar.
# El num_iterations optimo lo determina early stopping en cada evaluacion.
# CADA evaluacion se appendea al checkpoint_file antes de retornar.

Estimar_AUC_lightgbm <- function(learning_rate_log,
                                 num_leaves_log,
                                 feature_fraction,
                                 min_sum_hessian_log) {

  t0 <- Sys.time()

  # destransformo los que estan en escala log
  hp_variables <- list(
    learning_rate           = exp(learning_rate_log),
    num_leaves              = as.integer(round(exp(num_leaves_log))),
    feature_fraction        = feature_fraction,
    min_sum_hessian_in_leaf = exp(min_sum_hessian_log)
  )

  # combino con los fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, hp_variables)

  # entreno
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]
  niter <- modelo_train$best_iter
  dt_seg <- as.numeric(difftime(Sys.time(), t0, units="secs"))

  # --- CHECKPOINT INCREMENTAL: guardo esta evaluacion a disco YA ---
  fila <- data.table(
    ts = format(Sys.time(), "%Y-%m-%d %H:%M:%S"),
    learning_rate_log = learning_rate_log,
    num_leaves_log = num_leaves_log,
    feature_fraction = feature_fraction,
    min_sum_hessian_log = min_sum_hessian_log,
    learning_rate = hp_variables$learning_rate,
    num_leaves = hp_variables$num_leaves,
    min_sum_hessian_in_leaf = hp_variables$min_sum_hessian_in_leaf,
    num_iterations = niter,
    AUC = AUC,
    seg = round(dt_seg, 1)
  )
  fwrite(fila,
         file = PARAM$bo$checkpoint_file,
         sep = "\t",
         append = file.exists(PARAM$bo$checkpoint_file))

  message(format(Sys.time(), "%X  "),
    "lr=", round(hp_variables$learning_rate, 5),
    "  leaves=", hp_variables$num_leaves,
    "  ff=", round(feature_fraction, 3),
    "  msh=", round(hp_variables$min_sum_hessian_in_leaf, 4),
    "  niter=", niter,
    "  AUC=", round(AUC, 6),
    "  seg=", round(dt_seg, 0)
  )

  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return(list(Score = AUC, num_iterations = niter))
}

seteo de la Bayesian Optimization

In [ ]:
# --- Bounds de la Bayesian Optimization CONSERVADORA ---
# Cambios respecto del 9102 (que topo bounds y mostro sintomas de overfit):
#   num_leaves cap 1024 -> 256 (fuerza modelos menos capaces)
#   min_sum_hessian piso 1e-8 -> 1e-3 (fuerza regularizacion)
# Hipotesis: la BO del 9102 estaba pidiendo capacidad extrema porque
# el validation (202107) permitia overfit. Con estos bounds la BO
# tiene que encontrar el mejor modelo REGULARIZADO.

bo_bounds <- list(
  learning_rate_log    = c(log(0.001), log(1.5)),  # sin cambio
  num_leaves_log       = c(log(2),     log(256)),  # techo bajado
  feature_fraction     = c(0.05,       1.0),       # sin cambio
  min_sum_hessian_log  = c(log(1e-3),  log(1000))  # piso subido
)

# Mismo presupuesto que 9102
PARAM$bo$init_points <- 6
PARAM$bo$iters_bayes <- 30
# Total = 36 evaluaciones. Estimado: 1-2 hs (mas rapido que 9102 porque
# num_leaves chico entrena mas rapido).

##### Corrida de la Bayesian Optimization,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora entre 4 y 8 horas
<br> (48 evaluaciones = 8 initPoints + 40 iters bayesianas)

In [ ]:
# --- Corrida de la Bayesian Optimization ---
# 2-4 horas estimadas para 36 evaluaciones. Cada evaluacion se guarda a
# 'bo_history_incremental.txt' apenas termina, asi que si el kernel muere
# a mitad de camino, ese archivo tiene todo lo hecho hasta ese punto.

# === SANITY CHECK: 1 evaluacion con num_leaves cerca del tope (256) ===
cat("=== Sanity check: 1 evaluacion con num_leaves=256 ===\n")
t0 <- Sys.time()
resultado_test <- Estimar_AUC_lightgbm(
  learning_rate_log = log(0.05),
  num_leaves_log = log(256),
  feature_fraction = 0.5,
  min_sum_hessian_log = log(0.01)
)
t1 <- Sys.time()
gc_info <- gc()
dur_min <- as.numeric(difftime(t1, t0, units="mins"))
ram_mb <- round(sum(gc_info[, 6]))
cat("\nDuracion:", round(dur_min, 2), "min\n")
cat("RAM pico (Mb):", ram_mb, "\n")
cat("Referencia: en el 9102 con num_leaves=1024 tardo 1.6 min y uso 5.9 GB\n")
cat("Este con num_leaves=256 deberia tardar la mitad o menos.\n")

if (file.exists(PARAM$bo$checkpoint_file)) {
  file.remove(PARAM$bo$checkpoint_file)
  cat("\nCheckpoint del sanity check limpiado.\n")
}

In [ ]:
# --- Corrida efectiva de la BO (correr solo si el sanity check pasó) ---
set.seed(PARAM$semilla_primigenia)

bo_result <- bayesOpt(
  FUN         = Estimar_AUC_lightgbm,
  bounds      = bo_bounds,
  initPoints  = PARAM$bo$init_points,
  iters.n     = PARAM$bo$iters_bayes,
  iters.k     = 1,
  acq         = "ei",
  gsPoints    = 100,
  verbose     = 2,
  errorHandling = "continue"
)

# Persisto el resultado completo (redundante con el checkpoint pero por si acaso)
saveRDS(bo_result, "bo_result.rds")
cat("\nBO terminada. Resultado guardado en bo_result.rds\n")

la Bayesian Optimization ha corrido, extraigo los mejores hiperparametros

In [ ]:
# --- Extraigo el historial de la BO ---
# Prefiero el resultado en memoria (bo_result), pero si no esta, cargo
# desde el checkpoint incremental que se fue apendeando.

if (exists("bo_result") && !is.null(bo_result)) {
  tb_bo <- as.data.table(bo_result$scoreSummary)
  cat("Historial cargado desde bo_result en memoria:", nrow(tb_bo), "filas\n")
} else if (file.exists(PARAM$bo$checkpoint_file)) {
  tb_bo <- fread(PARAM$bo$checkpoint_file)
  # renombro para compatibilidad con extraccion (uso 'AUC' como Score)
  setnames(tb_bo, "AUC", "Score")
  cat("Historial cargado desde checkpoint incremental:", nrow(tb_bo), "filas\n")
} else {
  stop("No hay resultado de BO en memoria ni en disco.")
}

print(head(tb_bo[order(-Score)], 10))

fwrite(tb_bo, file= "bo_history.txt", sep="\t")

In [ ]:
# --- Extraigo la mejor combinacion de la BO ---
# Ordeno por Score (AUC) descendente, la 1ra fila es la mejor
setorder(tb_bo, -Score)
mejor_bo <- tb_bo[1]
print(mejor_bo)

# Destransformo desde la escala log a valores utilizables
PARAM$out$lgbm$AUC <- mejor_bo$Score
PARAM$out$lgbm$mejores_hiperparametros <- list(
  learning_rate           = exp(mejor_bo$learning_rate_log),
  num_leaves              = as.integer(round(exp(mejor_bo$num_leaves_log))),
  feature_fraction        = mejor_bo$feature_fraction,
  min_sum_hessian_in_leaf = exp(mejor_bo$min_sum_hessian_log),
  num_iterations          = mejor_bo$num_iterations
)

cat("\n--- MEJORES HIPERPARAMETROS ENCONTRADOS ---\n")
print(PARAM$out$lgbm$mejores_hiperparametros)
cat("AUC en validation:", round(PARAM$out$lgbm$AUC, 6), "\n")

In [ ]:
# el historial completo de la BO ya se imprimio arriba

### 9.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)


dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

##### Final Training Hyperparameters

In [ ]:
# uno los parametros fijos y los mejores encontrados por la BO
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que se decide en la BO / lo que ya no aplica al training final
fijos$num_iterations         <- NULL  # viene de la BO
fijos$early_stopping_rounds  <- NULL  # no hay validation en el final
fijos$metric                 <- NULL  # no evaluo durante el training final

# agrego a los hiperparametros fijos los que encontre con la BO
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

cat("--- HIPERPARAMETROS DEL SEMILLERIO FINAL ---\n")
print(param_final)

# --- SEMILLERIO k=20 ---
# 5 semillas originales + 15 primos nuevos (banco de 100 primos)
PARAM$semillerio$semillas <- c(
  # las 5 originales
  804043, 653561, 703903, 439693, 665857,
  # las 15 nuevas
  246319, 719179, 688511, 678859, 759179,
  748567, 319687, 771091, 684007, 514853,
  377749, 329977, 757927, 724837, 216973
)

stopifnot(length(PARAM$semillerio$semillas) == 20)
stopifnot(length(unique(PARAM$semillerio$semillas)) == 20)

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [ ]:
# --- LOOP DEL SEMILLERIO ---
# Para cada semilla:
#   1) entreno un LightGBM identico salvo por la semilla
#   2) predigo sobre el futuro (202109)
#   3) guardo la probabilidad individual a disco (para ensembles post-hoc)
#   4) acumulo la probabilidad en una tabla para promediar al final

# Preparo el dataset futuro UNA sola vez (fuera del loop)
PARAM$trainingstrategy$future <- c(202109)
dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

# Matriz de features del futuro, la calculo una vez y la reuso
mfuture <- data.matrix(dfuture[, campos_buenos, with= FALSE])

# Tabla acumuladora: una columna por semilla, con la probabilidad
tb_probs <- dfuture[, list(numero_de_cliente)]

# Carpeta para las predicciones individuales por semilla
dir.create("semillas", showWarnings= FALSE)

# Guardo el ultimo modelo entrenado para poder extraer importancia despues
final_model <- NULL

for (i in seq_along(PARAM$semillerio$semillas)) {

  semilla <- PARAM$semillerio$semillas[i]
  cat(format(Sys.time(), "%X"),
      " - Semilla ", i, "/", length(PARAM$semillerio$semillas),
      " = ", semilla, "\n", sep="")

  # piso la semilla en los hiperparametros
  param_semilla <- param_final
  param_semilla$seed <- semilla

  # entreno
  modelo_i <- lgb.train(
    data= dfinal_train,
    param= param_semilla,
    verbose= -100
  )

  # predigo sobre el futuro
  prob_i <- predict(modelo_i, mfuture)

  # nombre de la columna en la tabla acumuladora
  col_semilla <- paste0("prob_", semilla)
  tb_probs[, (col_semilla) := prob_i]

  # guardo la prediccion individual a disco (util para ensembles post-hoc
  # y para subir la semilla suelta a Kaggle y medir su ganancia)
  tb_pred_i <- dfuture[, list(numero_de_cliente)]
  tb_pred_i[, prob := prob_i]
  fwrite(tb_pred_i,
    file= paste0("semillas/prediccion_semilla_", semilla, ".txt"),
    sep= "\t"
  )

  # me quedo con el ultimo modelo entrenado (para importancia de vars)
  final_model <- modelo_i

  # limpio memoria
  rm(modelo_i, prob_i, tb_pred_i)
  gc(full= TRUE, verbose= FALSE)
}

cat("Semillerio completado. ", length(PARAM$semillerio$semillas),
    " modelos entrenados.\n", sep="")

In [ ]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(final_model, "modelo.txt")

In [ ]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(final_model))
archivo_importancia <- "impo.txt"

fwrite( tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

#### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
# dfuture ya fue definido antes del loop del semillerio

In [ ]:
# el predict se hace dentro del loop del semillerio

##### Tabla Prediccion

In [ ]:
# --- PROMEDIO DEL SEMILLERIO ---
# Promedio ARITMETICO de las probabilidades de las N semillas.
# Este es el ensemble: la probabilidad de baja+2 segun el semillerio.

cols_prob <- grep("^prob_", colnames(tb_probs), value= TRUE)
cat("Semillas en el ensemble: ", length(cols_prob), "\n")

tb_prediccion <- tb_probs[, list(numero_de_cliente)]
tb_prediccion[, prob := rowMeans(tb_probs[, ..cols_prob])]

# grabo el promedio del semillerio
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

# grabo tambien la tabla ancha con TODAS las probabilidades individuales
# util para armar ensembles con distintos subconjuntos de semillas post-hoc
fwrite(tb_probs,
  file= "probs_por_semilla.txt",
  sep= "\t"
)

# resumen de la correlacion entre semillas: si es muy alta (>0.99) el semillerio
# aporta menos varianza-reduction; si es mas baja, el ensemble ayuda mas
if (length(cols_prob) >= 2) {
  mat_probs <- as.matrix(tb_probs[, ..cols_prob])
  cor_matrix <- cor(mat_probs)
  cat("Correlacion media entre semillas: ",
      round(mean(cor_matrix[upper.tri(cor_matrix)]), 4), "\n")
}

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle

In [ ]:
# --- SUBMIT DEL ENSEMBLE (promedio del semillerio) ---
# Genero un archivo por cada corte y los subo a Kaggle.
# Este es el submit principal: mide la ganancia del semillerio completo.

PARAM$kaggle$competencia <- "data-mining-junior-2026-a"
PARAM$kaggle$cortes <- seq(1800, 2400, by = 100)

# ordeno por probabilidad promedio descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings= FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marco los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento,
                           "_BOcons_k20_", envios, ".csv")

  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste("-f", archivo_kaggle)

  mensaje <- paste0("-m 'BO conservador k=", length(PARAM$semillerio$semillas),
                    " envios=", envios, "'")

  linea <- paste(comando, competencia, arch, mensaje)
  salida <- system(linea, intern=TRUE)
  Sys.sleep(30)
  cat(salida, "\n")
}

# --- SUBMIT DE SEMILLAS INDIVIDUALES (OPCIONAL, off por default) ---
# Sirve para medir la ganancia Public/Private de CADA semilla por separado.
# Es lo que te da los 5 puntos por experimento para el test de Wilcoxon
# pareado cuando compares este baseline con futuras variantes (BO, FE, etc).
#
# CUIDADO: 5 semillas x 1 corte = 5 submits extra (10 en total con el ensemble).
# Si vas a hacer varias corridas en el mismo dia, considera el rate limit.
# Por eso subo solo UN corte por semilla (el central), no todos los cortes.

PARAM$kaggle$subir_individuales <- FALSE  # TRUE para activar
PARAM$kaggle$corte_individual <- 2100      # corte central del rango

if (PARAM$kaggle$subir_individuales) {
  for (semilla in PARAM$semillerio$semillas) {

    tb_ind <- fread(paste0("semillas/prediccion_semilla_", semilla, ".txt"))
    setorder(tb_ind, -prob)
    tb_ind[, Predicted := 0L]
    tb_ind[1:PARAM$kaggle$corte_individual, Predicted := 1L]

    archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento,
                             "_semilla_", semilla,
                             "_", PARAM$kaggle$corte_individual, ".csv")

    fwrite(tb_ind[, list(numero_de_cliente, Predicted)],
      file= archivo_kaggle, sep= ",")

    comando <- "kaggle competitions submit"
    competencia <- paste("-c", PARAM$kaggle$competencia)
    arch <- paste("-f", archivo_kaggle)
    mensaje <- paste0("-m 'semilla_individual=", semilla,
                      " envios=", PARAM$kaggle$corte_individual, "'")

    linea <- paste(comando, competencia, arch, mensaje)
    salida <- system(linea, intern=TRUE)
    Sys.sleep(30)
    cat(salida, "\n")
  }
}

In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")